In [ ]:
import requests
from dotenv import load_dotenv
import os

load_dotenv()

def post_message_to_slack(channel, text, token):
    """
    Sends a message to a specified Slack channel.
    
    Args:
        channel (str): The Slack channel to post the message to.
        text (str): The message text to send.
        token (str): The Slack API token for authentication.
    
    Returns:
        dict: The response JSON from the Slack API.
    """
    url = 'https://slack.com/api/chat.postMessage'
    headers = {
        'Content-type': 'application/json',
        'Authorization': f"Bearer {token}",
    }
    payload = {
        'channel': channel,
        'text': text
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()  # Raise an HTTPError for bad responses (4xx and 5xx)
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error posting message to Slack: {e}")
        return None

# Example usage
slack_token = os.getenv('SLACK_API_TOKEN')
channel_name = '#paper'
message_text = 'Hello, world!'

result = post_message_to_slack(channel_name, message_text, slack_token)
if result:
    print(result)

In [ ]:
from pprint import pprint

def fetch_qiita_stocked_articles(user_id, api_token):
    """
    Fetches the first stocked article for a Qiita user.
    
    Args:
        user_id (str): The Qiita user ID.
        api_token (str): The Qiita API token for authentication.
    
    Returns:
        list: The first article if available, otherwise an empty list.
    """
    url = f"https://qiita.com/api/v2/users/{user_id}/stocks"
    headers = {
        "Authorization": f"Bearer {api_token}"
    }
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        print(f"Error fetching data: {response.status_code}")
        return []
    
    articles = response.json()
    if isinstance(articles, list):
        return articles[0] if articles else []
    else:
        print(f"Error fetching data: {articles}")
        return []

# Fetch Qiita stocked articles
qiita_user_id = os.getenv('QIITA_USER_ID')
qiita_api_token = os.getenv('QIITA_API_TOKEN')

if qiita_user_id and qiita_api_token:
    qiita_result = fetch_qiita_stocked_articles(qiita_user_id, qiita_api_token)
    pprint(qiita_result)
else:
    print("Qiita user ID or API token is not set.")


In [ ]:
post_message_to_slack(channel_name, qiita_result['url'], slack_token)

In [ ]:
# requestは，stockが空ならば、空のリストを返すようになっている

# stockが空でない場合は，stockの中身を最初の要素を取り出す
# もし記事を残しておきたい場合はいいね，もう読まなくても良さそうならストックから削除するようにする
# 一度slackに送信した記事のidを保存しておく
# 保存したidと一致するものがあれば、スキップする
# 保存したidと一致するものがなければ、slackに送信する
# 送信したidを保存する

result[0].keys()

In [ ]:
import requests
from dotenv import load_dotenv
import os
from pprint import pprint
import json


load_dotenv()

NOTION_API_KEY = os.getenv('NOTION_API_KEY')
DATABASE_ID = os.getenv('NOTION_DATABASE_ID')

# リクエストヘッダー
headers = {
    "Authorization": f"Bearer {NOTION_API_KEY}",
    "Content-Type": "application/json",
    "Notion-Version": "2022-06-28",
}

# データベースに追加するデータ
data = {
    "parent": {"database_id": DATABASE_ID},
    "properties": {
        "Title": {
            "title": [
                {
                    "text": {
                        "content": "New Task"
                    }
                }
            ]
        },
    }
}

# データベースに新しいページを追加するAPIリクエスト
response = requests.post(
    "https://api.notion.com/v1/pages",
    headers=headers,
    data=json.dumps(data)
)

# レスポンスの確認
if response.status_code == 200:
    print("ページが正常に追加されました！")
else:
    print(f"エラーが発生しました: {response.status_code}")
    pprint(response.text)


In [ ]:
def fetch_titles_from_db(db_id, api_key):
    url = f"https://api.notion.com/v1/databases/{db_id}/query"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "Notion-Version": "2022-06-28",
    }
    
    try:
        # APIリクエストを送信
        response = requests.post(url, headers=headers)

        if response.status_code == 200:
            data = response.json()
            results = data.get('results', [])
            titles = [result.get('properties', {}).get('Title', {}).get('title', [])[0].get('plain_text', '') for result in results]
            return titles
        else:
            print(f"Error: {response.status_code}")
            print(response.text)
            return []
    except Exception as e:
        print(f"An error occurred: {e}")
        return []

In [ ]:
def is_title_in_notion_database(title: str, notion_token: str, database_id: str) -> bool:
    """
    Checks if a given title exists in a Notion database.

    Args:
        title (str): The title to search for in the database.
        notion_token (str): The Notion API token for authentication.
        database_id (str): The ID of the Notion database.

    Returns:
        bool: True if the title exists in the database, False otherwise.
    """
    url = f"https://api.notion.com/v1/databases/{database_id}/query"
    headers = {
        "Authorization": f"Bearer {notion_token}",
        "Content-Type": "application/json",
        "Notion-Version": "2022-06-28"
    }
    payload = {
        "filter": {
            "property": "Title",
            "title": {
                "contains": title
            }
        }
    }

    try:
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()
        data = response.json()
        return len(data.get("results", [])) > 0
    except requests.exceptions.RequestException as e:
        print(f"Error querying Notion database: {e}")
        return False


is_title_in_notion_database("New Task", NOTION_API_KEY, DATABASE_ID)

In [ ]:
import arxiv

search = arxiv.Search(
    query="machine learning",
    max_results=5,
    sort_by=arxiv.SortCriterion.Relevance
)

search.results()

In [ ]:
import requests
import xmltodict

def search_arxiv(query, max_results=5):
    base_url = "http://export.arxiv.org/api/query"
    params = {
        "search_query": f"all:{query}",
        "start": 0,
        "max_results": max_results
    }
    
    # APIにリクエストを送信
    response = requests.get(base_url, params=params)
    
    if response.status_code != 200:
        print("エラーレスポンス:", response.status_code)
        return

    # XMLをPythonの辞書形式に変換
    data = xmltodict.parse(response.content)
    
    # 結果の解析
    entries = data['feed'].get('entry', [])
    if isinstance(entries, dict):
        entries = [entries]  # 1件の場合は辞書として返ってくるためリスト化
    
    print(f"\n🔎 '{query}' に関する論文が {len(entries)} 件見つかりました。\n")
    for entry in entries:
        print(f"タイトル: {entry['title']}")
        authors = entry.get('author', [])
        if isinstance(authors, dict):
            authors = [authors]
        print(f"著者: {', '.join([author['name'] for author in authors])}")
        print(f"リンク: {entry['id']}")
        print("-" * 50)

    return entries

# クエリを指定して検索
entries = search_arxiv("machine learning")

In [ ]:
print(entries[0].keys())
entries[0]['summary'].replace('\n', ' ')

In [ ]:
from openai import OpenAI

client = OpenAI(
  api_key=os.getenv('OPENAI_API_KEY')
)

completion = client.chat.completions.create(
  model="gpt-4o-mini",
  store=True,
  messages=[
    {"role": "user", "content": "write a haiku about ai"}
  ]
)

print(completion.choices[0].message)


In [ ]:
completion

In [ ]:
# Qittaでストックした記事をSlackに送信する関数を定義
import os

import arxiv
from dotenv import load_dotenv
import requests


def fetch_paper_from_arxiv(query):
    """
    Fetches the first article from arXiv.
    
    Returns:
        dict: The first article if available, otherwise an empty dictionary.
    """
    search = arxiv.Search(
        query=query,
        max_results=1,
        sort_by=arxiv.SortCriterion.Relevance,
        sort_order=arxiv.SortOrder.Descending
    )
    articles = list(search.results())

    if articles:
        return articles[0]
    else:
        print(f"Error fetching data: {articles}")
        return {}


def post_message_to_slack(channel, text, token):
    """
    Sends a message to a specified Slack channel.
    
    Args:
        channel (str): The Slack channel to post the message to.
        text (str): The message text to send.
        token (str): The Slack API token for authentication.
    
    Returns:
        dict: The response JSON from the Slack API.
    """
    url = 'https://slack.com/api/chat.postMessage'
    headers = {
        'Content-type': 'application/json',
        'Authorization': f"Bearer {token}",
    }
    payload = {
        'channel': channel,
        'text': text
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()  # Raise an HTTPError for bad responses (4xx and 5xx)
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error posting message to Slack: {e}")
        return None


load_dotenv()


slack_token = os.getenv('SLACK_API_TOKEN')

channel_name = '#paper'
query = "Personalized Federated Learning"
paper_result = fetch_paper_from_arxiv(query)
if paper_result:
    message = f'今日の論文はこちら: 「{paper_result.title}」\n{paper_result.pdf_url}'
else:
    message = "論文が見つかりませんでした．"
post_message_to_slack(channel_name, message, slack_token)


In [ ]:
paper_result.summary

In [ ]:
import openai

openai.api_key = os.getenv('OPENAI_API_KEY')

def get_summary(result):
    system = """与えられた論文の要点を3点のみでまとめ、以下のフォーマットで日本語で出力してください。```
    タイトルの日本語訳
    ・要点1
    ・要点2
    ・要点3
    ```"""

    text = f"title: {result.title}\nbody: {result.summary}"
    response = openai.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {'role': 'system', 'content': system},
                    {'role': 'user', 'content': text}
                ],
                temperature=0.25,
            )
    response = response.model_dump()
    summary = response['choices'][0]['message']['content']
    # schemaを定義するのもいいかも

    # ---メッセージを作成する---
    title_en = result.title
    title, *body = summary.split('\n')
    body = '\n'.join(body)
    date_str = result.published.strftime("%Y-%m-%d %H:%M:%S")
    message = f"発行日: {date_str}\n{result.entry_id}\n{title_en}\n{title}\n{body}\n"
    # ----------------------
    
    return message

In [ ]:
result = paper_result

system = """与えられた論文の要点を3点のみでまとめ、以下のフォーマットで日本語で出力してください。```
タイトルの日本語訳
・要点1
・要点2
・要点3
```"""

text = f"title: {result.title}\nbody: {result.summary}"
response = openai.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {'role': 'system', 'content': system},
                {'role': 'user', 'content': text}
            ],
            temperature=0.25,
        )


In [ ]:
response = response.model_dump()

summary = response['choices'][0]['message']['content']
title_en = result.title
title, *body = summary.split('\n')
body = '\n'.join(body)
date_str = result.published.strftime("%Y-%m-%d %H:%M:%S")
message = f"発行日: {date_str}\n{result.entry_id}\n{title_en}\n{title}\n{body}\n"

In [ ]:
pprint(message)

In [ ]:
from pprint import pprint
pprint(summary)

In [ ]:
test_response

In [ ]:
notion_token = os.getenv('NOTION_API_KEY')
database_id = os.getenv('NOTION_DATABASE_ID')
title = "New Task"


url = f"https://api.notion.com/v1/pages"
headers = {
    "Authorization": f"Bearer {notion_token}",
    "Content-Type": "application/json",
    "Notion-Version": "2022-06-28",
}
data = {
    "parent": {"database_id": database_id},
    "properties": {
        "Title": {
            "title": [
                {
                    "text": {
                        "content": title
                    }
                }
            ]
        },
        "Abstract": {
            "rich_text": [
                {
                    "type": "text",
                    "text": {
                        "content": summary,
                    }
                }
            ]
        },
        "Status": {
            "status": {
                "name": "To Read",
                "color": "gray"
            }
        },
        # "URL": {
        # },
        # "Notes": {
        # },
    }
}
response = requests.post(
    "https://api.notion.com/v1/pages",
    headers=headers,
    data=json.dumps(data)
)

In [ ]:
'''
Arxivの論文を取得して、要約を生成し、Slackに送信し、Notionに保存する一連の流れを実装しました。
- Arxivから論文を取得: 関数名はfetch_paper_from_arxiv
- OpenAIを使って要約を生成: 関数名はget_summary
- Slackに要約を送信: 関数名はpost_message_to_slack
- Notionに要約を保存: APIリクエストを送信
'''

In [ ]:
def get_page_id(title: str, database_id: str) -> str:
    """
    タイトルからページIDを取得
    Args:
        title: タイトル
        database_id: データベースID
    Returns:
        page_id: ページID
    """
    # タイトルを検索して，ページIDを取得
    url = f"https://api.notion.com/v1/databases/{database_id}/query"
    payload = {"filter": {"property": "Title", "rich_text": {"equals": title}}}
    response = requests.post(url, json=payload, headers=headers)
    # pprint(response.json())
    page_id = response.json()["results"][0]["id"]
    return page_id

get_page_id("Leveraging Learning Metrics for Improved Federated Learning", database_id)

In [ ]:
page_id = get_page_id("Leveraging Learning Metrics for Improved Federated Learning", database_id)

payload = {
    "children": [
        {
            "paragraph": {
                "rich_text": [
                    { 
                        "text": {
                            "content": "This is a test."
                        }
                    }
                ]
            }
        }
    ]
}

url = f"https://api.notion.com/v1/blocks/{page_id}/children"
response = requests.patch(url, json=payload, headers=headers)
response.json()